# ELT — CPGF: Extract & Load

Ler os CSVs brutos de **2023, 2024 e 2025** e carregá-los em tabelas **Staging/Raw**, sem transformação prévia.

Usamos `csv` e `sqlite3`, da biblioteca padrão, para ler em lotes sem inferência de tipos e controlar a transação dos três anos.

Cada coluna mantém seu nome original e é armazenada como `TEXT`. Preservamos zeros à esquerda, máscaras, espaços, campos vazios (`''`), datas, vírgulas decimais, marcadores de sigilo e registros repetidos. Não há limpeza, renomeação, deduplicação ou conversão de valores. A transformação em dimensões e fatos fica para uma etapa posterior, dentro do banco.

## 1. Preparando o ambiente

Use Python 3.9 ou mais recente; não é necessário instalar pandas, SQLAlchemy ou dbt para esta etapa. Execute as células na ordem, preferencialmente com **Reiniciar kernel e executar tudo**.

Os arquivos anuais devem existir em `data/raw/`. Caso ainda não existam, execute `python scripts/extract_cpgf.py` na raiz do repositório (no Colab, use `!python scripts/extract_cpgf.py`). Esse script reúne os arquivos mensais em CSVs UTF-8 separados por `;`; aqui, a extração consiste na leitura desses CSVs.

No Colab, clone o repositório e configure `RAIZ_CONFIGURADA`. Para experimentar apenas os arquivos versionados, use `MODO_AMOSTRA = True`; o resultado ficará em outro banco.

In [ ]:
from pathlib import Path
from itertools import islice, zip_longest
import csv
import sqlite3
import time

RAIZ_CONFIGURADA = None  # No Colab: Path('/content/projeto-integracao-dw')
MODO_AMOSTRA = False
ANOS = (2023, 2024, 2025)
TAMANHO_LOTE = 10_000


def localizar_raiz(configurada=None):
    candidatos = [Path(configurada)] if configurada is not None else [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / 'scripts/extract_cpgf.py').is_file():
            return pasta.resolve()
    raise FileNotFoundError('Configure RAIZ_CONFIGURADA com a pasta do repositório.')


RAIZ = localizar_raiz(RAIZ_CONFIGURADA)
arquivos_brutos = {
    f'raw_cpgf_{ano}': RAIZ / 'data/raw' / (
        f'amostras/{ano}_amostra.csv' if MODO_AMOSTRA else f'{ano}_CPGF.csv'
    )
    for ano in ANOS
}
DESTINO = RAIZ / 'data/database' / (
    'cpgf_elt_amostra.sqlite' if MODO_AMOSTRA else 'cpgf_elt.sqlite'
)
print('Modo:', 'amostra' if MODO_AMOSTRA else 'completo')
print('Banco de destino:', DESTINO)

## 2. Definindo a leitura dos dados brutos

O SQLite não possui um schema `raw` separado neste arquivo: o prefixo `raw_` identifica a camada. Criamos uma tabela para cada arquivo anual, com as mesmas 15 colunas da fonte, sem colunas adicionais ou restrições de unicidade.

O leitor CSV retorna strings, inclusive para campos vazios e textos como `NA` e `NULL`. A leitura em lotes limita o uso de memória. Cabeçalhos inesperados, registros com quantidade incorreta de campos e erros de leitura interrompem a carga; nenhuma linha é ignorada silenciosamente. Essas conferências validam a estrutura, sem alterar o conteúdo.

In [ ]:
COLUNAS = [
    'CÓDIGO ÓRGÃO SUPERIOR', 'NOME ÓRGÃO SUPERIOR',
    'CÓDIGO ÓRGÃO', 'NOME ÓRGÃO',
    'CÓDIGO UNIDADE GESTORA', 'NOME UNIDADE GESTORA',
    'ANO EXTRATO', 'MÊS EXTRATO', 'CPF PORTADOR', 'NOME PORTADOR',
    'CNPJ OU CPF FAVORECIDO', 'NOME FAVORECIDO', 'TRANSAÇÃO',
    'DATA TRANSAÇÃO', 'VALOR TRANSAÇÃO',
]


def identificador(nome):
    # Aspas SQL permitem usar os nomes originais, inclusive espaços e acentos.
    return '"' + nome.replace('"', '""') + '"'


def ler_registros(caminho):
    with Path(caminho).open(encoding='utf-8-sig', newline='') as arquivo:
        leitor = csv.reader(arquivo, delimiter=';', strict=True)
        if next(leitor, None) != COLUNAS:
            raise ValueError(f'{caminho.name}: cabeçalho diferente do esperado.')
        for numero, registro in enumerate(leitor, start=1):
            if len(registro) != len(COLUNAS):
                raise ValueError(
                    f'{caminho.name}: registro {numero} tem {len(registro)} campos; '
                    f'esperados {len(COLUNAS)}.'
                )
            yield tuple(registro)


def ler_lotes(caminho, tamanho):
    if tamanho <= 0:
        raise ValueError('O tamanho do lote precisa ser positivo.')
    registros = ler_registros(caminho)
    try:
        while True:
            lote = list(islice(registros, tamanho))
            if not lote:
                break
            yield lote
    finally:
        registros.close()

## 3. Conferindo os arquivos de entrada

Todos os arquivos precisam estar disponíveis antes da carga. A conferência inicial lê o cabeçalho e o primeiro registro; as demais linhas serão verificadas durante a leitura completa. Arquivos sem registros também interrompem a execução.

In [ ]:
for tabela, caminho in arquivos_brutos.items():
    if not caminho.is_file():
        raise FileNotFoundError(
            f'Arquivo ausente: {caminho}. Execute scripts/extract_cpgf.py '
            'ou habilite MODO_AMOSTRA para usar as amostras.'
        )
    registros = ler_registros(caminho)
    try:
        if next(registros, None) is None:
            raise ValueError(f'{caminho.name}: arquivo sem registros.')
    finally:
        registros.close()
    print(f'{caminho.name} → {tabela} ({caminho.stat().st_size:,} bytes)')

## 4. Carregando a camada Raw

A carga é completa: as três tabelas Raw são recriadas em **uma única transação**. Reexecutar o notebook atualiza seu conteúdo, sem acumular cópias. Outras tabelas no banco permanecem intactas.

Antes do commit, comparamos a contagem e **cada campo de cada registro** armazenado com os CSVs, na ordem de inserção (`rowid` interno do SQLite). Isso confere também vazios, espaços e duplicatas. Se qualquer leitura, inserção ou conferência falhar, o rollback preserva as tabelas da carga anterior. Mantenha os CSVs inalterados durante a execução.

In [ ]:
def validar_tabela(conexao, tabela, caminho, quantidade):
    nome_sql = identificador(tabela)
    colunas_sql = ', '.join(map(identificador, COLUNAS))
    total_banco = conexao.execute(f'SELECT COUNT(*) FROM {nome_sql}').fetchone()[0]
    if total_banco != quantidade:
        raise ValueError(f'{tabela}: contagem divergente após a carga.')

    cursor = conexao.execute(f'SELECT {colunas_sql} FROM {nome_sql} ORDER BY rowid')
    origem = ler_registros(caminho)
    ausente = object()
    try:
        for numero, (esperado, armazenado) in enumerate(
            zip_longest(origem, cursor, fillvalue=ausente), start=1
        ):
            if esperado != armazenado:
                raise ValueError(f'{tabela}: conteúdo divergente no registro {numero}.')
    finally:
        origem.close()
        cursor.close()
    return total_banco


def carregar_raw(arquivos, destino, tamanho_lote=TAMANHO_LOTE):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    colunas_sql = ', '.join(map(identificador, COLUNAS))
    definicao_sql = ', '.join(f'{identificador(coluna)} TEXT' for coluna in COLUNAS)
    parametros = ', '.join('?' for _ in COLUNAS)
    resumo = []
    conexao = sqlite3.connect(destino)
    try:
        # BEGIN explícito inclui DROP/CREATE na mesma transação dos INSERTs.
        with conexao:
            conexao.execute('BEGIN')
            for tabela, caminho in arquivos.items():
                nome_sql = identificador(tabela)
                conexao.execute(f'DROP TABLE IF EXISTS {nome_sql}')
                conexao.execute(f'CREATE TABLE {nome_sql} ({definicao_sql})')
                quantidade = 0
                for lote in ler_lotes(caminho, tamanho_lote):
                    conexao.executemany(
                        f'INSERT INTO {nome_sql} ({colunas_sql}) VALUES ({parametros})', lote
                    )
                    quantidade += len(lote)
                if quantidade == 0:
                    raise ValueError(f'{caminho.name}: arquivo sem registros.')
                total_banco = validar_tabela(conexao, tabela, caminho, quantidade)
                resumo.append({
                    'tabela': tabela, 'arquivo': caminho.name,
                    'linhas_csv': quantidade, 'linhas_banco': total_banco,
                })
    finally:
        conexao.close()
    return resumo

In [ ]:
inicio = time.perf_counter()
resumo_carga = carregar_raw(arquivos_brutos, DESTINO)
print('Carga Raw concluída e validada:', DESTINO)
print('Tempo de carga e validação (s):', round(time.perf_counter() - inicio, 2))

## 5. Conferindo o resultado

As contagens abaixo vêm da execução atual e da consulta ao banco, sem usar totais fixos. A prévia consulta diretamente os campos originais, ainda como texto. O banco completo fica em `data/database/cpgf_elt.sqlite`; o modo amostra usa `cpgf_elt_amostra.sqlite`.

In [ ]:
print(f'{"Tabela":<20} {"Linhas CSV":>12} {"Linhas banco":>14}')
for item in resumo_carga:
    print(f'{item["tabela"]:<20} {item["linhas_csv"]:>12,} {item["linhas_banco"]:>14,}')
print('Total carregado:', sum(item['linhas_banco'] for item in resumo_carga))

conexao = sqlite3.connect(f'{DESTINO.resolve().as_uri()}?mode=ro', uri=True)
try:
    conexao.row_factory = sqlite3.Row
    for tabela in arquivos_brutos:
        print(f'\nPrévia de {tabela}:')
        consulta = (
            'SELECT "ANO EXTRATO", "MÊS EXTRATO", "DATA TRANSAÇÃO", "VALOR TRANSAÇÃO" '
            f'FROM {identificador(tabela)} ORDER BY rowid LIMIT 3'
        )
        for registro in conexao.execute(consulta):
            print(dict(registro))
finally:
    conexao.close()